# 07. Расширенный канал mobile/ads: пять вариантов К. Азимовой в ансамбле с лучшими моделями других доменов

**Идея.** В пакете К. Азимовой опубликована одна модель на едином командном тесте — Exp-Combined (614 признаков). Четыре других её ноутбука (`baseline_run.ipynb`, `improved_counterfeit_v3_run.ipynb`, `kl_divergence_run.ipynb`, `paper_adaptation_run.ipynb`) обучены на её индивидуальном split-е (test = 19 818) с жёстко закреплённым assert-ом, что делает соответствующие probas несовместимыми со стекингом на team-test (58 410). Настоящий ноутбук, выполняя общекомандную интеграционную задачу, переобучает пять её вариантов на team-split-е, после чего каждый вариант объединяется через L1 Logistic Regression с лучшими headline-моделями трёх других доменов (M2-FE+ Мкоян, Fusion Красовской, MMD-Thinker v4 Бахтиаровой) в четырёхканальный ансамбль. Финальное сравнение пяти ансамблей выявляет, какой вариант mobile/ads-канала даёт оптимальный вклад в общекомандную модель.

**Воспроизводимость.** random_state = 42 везде. Полный Run-all на M4 Pro занимает ориентировочно 25–40 мин (основная стоимость — обучение Doc2Vec PV-DM в варианте K5).

## 0. Импорты и загрузка общих артефактов

In [1]:
from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from catboost import CatBoostClassifier
warnings.filterwarnings('ignore')

ROOT = Path('.')
PKG = ROOT / 'packages'
DIANA_FOLDER = ROOT / "data"
SERVICE = ROOT / 'counterfeit_service'
NB_DIR = ROOT / 'real_estate_approaches' / 'notebooks'
SEED = 42

def rap90(y_true, p):
    prec, rec, _ = precision_recall_curve(y_true, p)
    m = prec >= 0.9
    return float(rec[m].max()) if m.any() else 0.0

def metrics(y_true, p):
    return {'ROC-AUC': roc_auc_score(y_true, p),
            'PR-AUC':  average_precision_score(y_true, p),
            'R@P>=0.9': rap90(y_true, p)}

In [2]:
# Загрузка ozon_train.csv и team_split
df = pd.read_csv(DIANA_FOLDER / 'ml_ozon_ounterfeit_train.csv', encoding='utf-8', engine='python')
print(f'df shape: {df.shape}, контрафакт: {df["resolution"].mean():.4f}')

train_idx = np.load(PKG / 'package_diana' / '3_probas_team_split' / 'team_train_idx.npy')
val_idx   = np.load(PKG / 'package_diana' / '3_probas_team_split' / 'team_val_idx.npy')
test_idx  = np.load(PKG / 'package_diana' / '3_probas_team_split' / 'team_test_idx.npy')
y_test    = np.load(PKG / 'package_diana' / '3_probas_team_split' / 'y_test_team.npy').astype(np.int8)
print(f'train {len(train_idx)}, val {len(val_idx)}, test {len(test_idx)}')

y_train = df.iloc[train_idx]['resolution'].astype('int8').values
y_val   = df.iloc[val_idx]['resolution'].astype('int8').values
assert np.array_equal(df.iloc[test_idx]['resolution'].astype('int8').values, y_test)
print(f'positive rate: train {y_train.mean():.4f}, val {y_val.mean():.4f}, test {y_test.mean():.4f}')
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'scale_pos_weight = {scale_pos_weight:.2f}')

df shape: (197198, 45), контрафакт: 0.0662
train 69453, val 69335, test 58410
positive rate: train 0.0610, val 0.0602, test 0.0795
scale_pos_weight = 15.40


In [3]:
# Готовые probas лучших моделей трёх других авторов
p_diana_M2_FE_plus = np.load(PKG / 'package_diana'  / '3_probas_team_split' / 'test_proba_diana_team.npy').astype(np.float64)
p_soni_Fusion      = np.load(PKG / 'package_soni'   / '3_probas_team_split' / 'test_proba_fusion_team.npy').astype(np.float64)
p_albina_MMD_v4    = np.load(PKG / 'package_albina' / '3_probas_team_split' / 'test_proba_albina_team.npy').astype(np.float64)
print('Headline моделей трёх других доменов:')
for nm, p in [('Diana M2-FE+', p_diana_M2_FE_plus),
               ('Soni Fusion',  p_soni_Fusion),
               ('Albina MMD v4',p_albina_MMD_v4)]:
    m = metrics(y_test, p)
    print(f'  {nm:>16}: ROC={m["ROC-AUC"]:.4f} PR={m["PR-AUC"]:.4f} R@P={m["R@P>=0.9"]:.4f}')

Headline моделей трёх других доменов:
      Diana M2-FE+: ROC=0.9484 PR=0.7375 R@P=0.1154
       Soni Fusion: ROC=0.9522 PR=0.7284 R@P=0.1077
     Albina MMD v4: ROC=0.9545 PR=0.7084 R@P=0.0276


## 1. K1 — TF-IDF + Logistic Regression (текстовый baseline К. Азимовой)

Точная адаптация baseline-варианта из `baseline_run.ipynb` на команд-сплит. Текст склеивается из brand_name + description + name_rus. TF-IDF: n-grams (1, 2), max_features = 50 000, min_df = 5. Классификатор: LogisticRegression(class_weight='balanced', max_iter=1000).

In [4]:
def build_text(row):
    parts = []
    for col in ['brand_name', 'description', 'name_rus']:
        v = row.get(col)
        if pd.notna(v) and str(v).strip() and str(v) != 'nan':
            parts.append(str(v))
    return ' '.join(parts) if parts else 'empty'

df['text'] = df.apply(build_text, axis=1)
train_df, val_df, test_df = df.iloc[train_idx], df.iloc[val_idx], df.iloc[test_idx]
print(f'text built; example: {train_df["text"].iloc[0][:100]!r}')

text built; example: 'ACTRUM Мешки пылесборники для пылесоса PHILIPS, 10 шт., синтетические, многослойные, бренд: ACTRUM, '


In [5]:
t0 = time.time()
vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=5)
X_tf_train = vec.fit_transform(train_df['text'])
X_tf_test  = vec.transform(test_df['text'])
lr_text = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_text.fit(X_tf_train, y_train)
p_K1_tfidf_lr = lr_text.predict_proba(X_tf_test)[:, 1]
print(f'K1 fit+predict: {time.time()-t0:.0f}s')
print(f'K1 metrics: {metrics(y_test, p_K1_tfidf_lr)}')

K1 fit+predict: 6s
K1 metrics: {'ROC-AUC': 0.936252990459096, 'PR-AUC': 0.5860239939336281, 'R@P>=0.9': 0.007753607581305191}


## 2. K2 — CatBoost на 38 табличных признаках (без текста)

Командный pure-tabular baseline по drop-list из общекомандного `notebooks/baseline.ipynb`. CatBoost: iterations = 700, depth = 6, lr = 0.05, eval_metric = AUC, scale_pos_weight = N0/N1.

In [6]:
drop_cols = ['id', 'ItemID', 'SellerID', 'name_rus', 'description', 'brand_name', 'text', 'resolution']
tab_cols = [c for c in df.columns if c not in drop_cols]
cat_cols_tab = [c for c in tab_cols if df[c].dtype == 'object']
print(f'{len(tab_cols)} tab features; cat: {cat_cols_tab}')
X_tab_train = train_df[tab_cols].copy(); X_tab_val = val_df[tab_cols].copy(); X_tab_test = test_df[tab_cols].copy()
for c in cat_cols_tab:
    X_tab_train[c] = X_tab_train[c].fillna('nan').astype(str)
    X_tab_val[c]   = X_tab_val[c].fillna('nan').astype(str)
    X_tab_test[c]  = X_tab_test[c].fillna('nan').astype(str)
for c in [c for c in tab_cols if c not in cat_cols_tab]:
    X_tab_train[c] = X_tab_train[c].fillna(0)
    X_tab_val[c]   = X_tab_val[c].fillna(0)
    X_tab_test[c]  = X_tab_test[c].fillna(0)

38 tab features; cat: ['CommercialTypeName4']


In [7]:
t0 = time.time()
cb_tab = CatBoostClassifier(iterations=700, depth=6, learning_rate=0.05, eval_metric='AUC',
                             scale_pos_weight=scale_pos_weight, random_seed=SEED, verbose=0,
                             early_stopping_rounds=50)
cb_tab.fit(X_tab_train, y_train, cat_features=cat_cols_tab, eval_set=(X_tab_val, y_val), use_best_model=True)
p_K2_cb_tab = cb_tab.predict_proba(X_tab_test)[:, 1]
print(f'K2 fit: {time.time()-t0:.0f}s, best_iter={cb_tab.best_iteration_}')
print(f'K2 metrics: {metrics(y_test, p_K2_cb_tab)}')

K2 fit: 4s, best_iter=293
K2 metrics: {'ROC-AUC': 0.9049868919138462, 'PR-AUC': 0.5301273145500932, 'R@P>=0.9': 0.002153779883695886}


## 3. K3 — Image embeddings 32-dim + LR (визуальный baseline)

Адаптация из `baseline_run.ipynb`. Старые 32-мерные image embeddings из `counterfeit_service/image_embeddings.csv`. StandardScaler fit на train, LogisticRegression(class_weight='balanced').

In [8]:
import numpy as np
img_emb_raw = pd.read_csv(SERVICE / 'image_embeddings.csv', on_bad_lines='skip')
img_emb_raw.columns = [c.lower() for c in img_emb_raw.columns]
img_emb_raw['item_id'] = img_emb_raw['image_name'].str.replace('.png', '', regex=False).astype(int)
# 'embedding' column хранит строку '[a b c ...]' — распарсить в float32 вектор
def parse_emb(s):
    s = s.strip().lstrip('[').rstrip(']')
    return np.fromstring(s, sep=' ', dtype=np.float32)
emb_arr = np.stack([parse_emb(s) for s in img_emb_raw['embedding'].values])
img_dim = emb_arr.shape[1]
print(f'image_emb_raw: {img_emb_raw.shape}, parsed embedding dim = {img_dim}')
# Make a DataFrame indexed by ItemID for fast join
img_emb = pd.DataFrame(emb_arr, columns=[f'img_{i}' for i in range(img_dim)])
img_emb['item_id'] = img_emb_raw['item_id'].values
emb_cols = [f'img_{i}' for i in range(img_dim)]
print(f'img_emb final: {img_emb.shape}, {len(emb_cols)} feature cols')

image_emb_raw: (219195, 4), parsed embedding dim = 32
img_emb final: (219195, 33), 32 feature cols


In [9]:
def get_img_features(df_part, img_emb, emb_cols):
    merged = df_part[['id']].merge(img_emb[['item_id'] + emb_cols], left_on='id', right_on='item_id', how='left')
    X = merged[emb_cols].values.astype(np.float32)
    X = np.nan_to_num(X, nan=0.0)
    return X

X_img_train = get_img_features(train_df, img_emb, emb_cols)
X_img_test  = get_img_features(test_df,  img_emb, emb_cols)
print(f'X_img_train: {X_img_train.shape}, X_img_test: {X_img_test.shape}')
scaler_img = StandardScaler()
X_img_train_sc = scaler_img.fit_transform(X_img_train)
X_img_test_sc  = scaler_img.transform(X_img_test)
t0 = time.time()
lr_img = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_img.fit(X_img_train_sc, y_train)
p_K3_img_lr = lr_img.predict_proba(X_img_test_sc)[:, 1]
print(f'K3 fit: {time.time()-t0:.0f}s')
print(f'K3 metrics: {metrics(y_test, p_K3_img_lr)}')

X_img_train: (69453, 32), X_img_test: (58410, 32)
K3 fit: 0s
K3 metrics: {'ROC-AUC': 0.4962532353715926, 'PR-AUC': 0.07842459354457623, 'R@P>=0.9': 0.0}


## 4. K4 — Exp-Combined (614 признаков, готовая модель из `package_karina/3_probas_team_split/`)

Это та модель, которая К. Азимова уже опубликовала на едином командном тесте: feature fusion 38 tab + 512 CLIP + 50 TF-IDF SVD + 4 Deng-typosquatting + 11 FADAML-price/text. Берётся как есть.

In [10]:
p_K4_exp_combined = np.load(PKG / 'package_karina' / '3_probas_team_split' / 'test_proba_karina_team.npy').astype(np.float64)
print(f'K4 (готовая Exp-Combined) metrics: {metrics(y_test, p_K4_exp_combined)}')

K4 (готовая Exp-Combined) metrics: {'ROC-AUC': 0.9407216100360062, 'PR-AUC': 0.6562435802853975, 'R@P>=0.9': 0.06655179840620289}


## 5. K5 — Doc2Vec PV-DM + LR (paper-adaptation Karunanayake-style)

Воспроизведение метода из `paper_adaptation_run.ipynb`: PV-DM на текстах товаров, 100-мерные инференс-векторы, LR-классификатор. Параметры Doc2Vec: vector_size = 100, window = 5, min_count = 5, epochs = 10. Это самая дорогая по времени модель ноутбука (обучение Doc2Vec ~15 мин).

In [11]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
t0 = time.time()
tagged = [TaggedDocument(words=t.lower().split(), tags=[str(i)]) for i, t in enumerate(train_df['text'].values)]
d2v = Doc2Vec(vector_size=100, window=5, min_count=5, workers=8, epochs=10, dm=1, seed=SEED)
d2v.build_vocab(tagged)
d2v.train(tagged, total_examples=d2v.corpus_count, epochs=d2v.epochs)
print(f'Doc2Vec train: {time.time()-t0:.0f}s; vocab={len(d2v.wv)}')

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Doc2Vec train: 18s; vocab=48970


In [12]:
t0 = time.time()
X_d2v_train = np.array([d2v.dv[str(i)] for i in range(len(train_df))], dtype=np.float32)
X_d2v_test  = np.array([d2v.infer_vector(t.lower().split(), epochs=50) for t in test_df['text'].values], dtype=np.float32)
print(f'Doc2Vec infer test: {time.time()-t0:.0f}s')
lr_d2v = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_d2v.fit(X_d2v_train, y_train)
p_K5_doc2vec_lr = lr_d2v.predict_proba(X_d2v_test)[:, 1]
print(f'K5 metrics: {metrics(y_test, p_K5_doc2vec_lr)}')

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Doc2Vec infer test: 120s


K5 metrics: {'ROC-AUC': 0.7471780914948144, 'PR-AUC': 0.1875849472927503, 'R@P>=0.9': 0.0}


## 6. Пять четырёхканальных ансамблей через L1 Logistic Regression

Для каждого из пяти вариантов канала mobile/ads строится отдельный ансамбль с тремя headline-моделями других доменов: M2-FE+ (недвижимость), Fusion (финтех), MMD-Thinker v4 (соц. сети). Мета-классификатор — L1 LR(C, class_weight='balanced'), C выбирается по PR-AUC на meta-eval из сетки {0.01, 0.1, 1, 10}. Разбиение test — group-aware по SellerID (test_size=0.5, random_state=42).

In [13]:
sellers = np.load(NB_DIR / 'team_test_sellers.npy')
gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
fit_idx, eval_idx = next(gss.split(np.zeros((len(y_test), 1)), y_test, groups=sellers))
print(f'meta-fit {len(fit_idx)}, meta-eval {len(eval_idx)}, seller overlap = {len(set(sellers[fit_idx]) & set(sellers[eval_idx]))}')

def fit_ensemble(p_karina, label):
    X4 = np.column_stack([p_karina, p_diana_M2_FE_plus, p_soni_Fusion, p_albina_MMD_v4])
    sc = StandardScaler().fit(X4[fit_idx])
    best = None
    for C in [0.01, 0.1, 1.0, 10.0]:
        lr = LogisticRegression(C=C, class_weight='balanced', max_iter=2000, random_state=SEED)
        lr.fit(sc.transform(X4[fit_idx]), y_test[fit_idx])
        p_e = lr.predict_proba(sc.transform(X4[eval_idx]))[:, 1]
        m = metrics(y_test[eval_idx], p_e)
        if best is None or m['PR-AUC'] > best['m_eval']['PR-AUC']:
            best = {'C': C, 'm_eval': m, 'lr': lr, 'sc': sc, 'X4': X4}
    p_full = best['lr'].predict_proba(best['sc'].transform(best['X4']))[:, 1]
    m_full = metrics(y_test, p_full)
    coefs = best['lr'].coef_[0]
    return {'label': label, 'best_C': best['C'], 'metrics_eval': best['m_eval'], 'metrics_full': m_full,
            'coefs': {'karina': float(coefs[0]), 'diana_M2_FE_plus': float(coefs[1]),
                       'soni_Fusion': float(coefs[2]), 'albina_MMD_v4': float(coefs[3])},
            'p_full': p_full}

variants = [
    ('K1_TFIDF_LR',         p_K1_tfidf_lr),
    ('K2_CatBoost_38tab',   p_K2_cb_tab),
    ('K3_ImageEmb32_LR',    p_K3_img_lr),
    ('K4_ExpCombined_614',  p_K4_exp_combined),
    ('K5_Doc2Vec_LR',       p_K5_doc2vec_lr),
]
ensembles = [fit_ensemble(p, lbl) for (lbl, p) in variants]

rows = []
for e in ensembles:
    rows.append({
        'mobile/ads канал': e['label'],
        'best_C': e['best_C'],
        'PR (eval)': e['metrics_eval']['PR-AUC'],
        'R@P (eval)': e['metrics_eval']['R@P>=0.9'],
        'PR (full)': e['metrics_full']['PR-AUC'],
        'R@P (full)': e['metrics_full']['R@P>=0.9'],
        'ROC (full)': e['metrics_full']['ROC-AUC'],
    })
df_ens = pd.DataFrame(rows)
df_ens.round(4)

meta-fit 33568, meta-eval 24842, seller overlap = 0


,mobile/ads канал,best_C,PR (eval),R@P (eval),PR (full),R@P (full),ROC (full)
0,K1_TFIDF_LR,0.01,0.7446,0.1963,0.7463,0.1144,0.9599
1,K2_CatBoost_38tab,0.01,0.7346,0.1237,0.7517,0.1803,0.9582
2,K3_ImageEmb32_LR,0.01,0.7382,0.2769,0.7464,0.1736,0.9579
3,K4_ExpCombined_614,0.01,0.7385,0.2283,0.7445,0.1643,0.9583
4,K5_Doc2Vec_LR,0.01,0.7349,0.2566,0.7468,0.1807,0.9569


### Коэффициенты L1 LR для каждого ансамбля

In [14]:
rows = []
for e in ensembles:
    rows.append({'канал mobile/ads': e['label'], **e['coefs']})
pd.DataFrame(rows).round(4)

,канал mobile/ads,karina,diana_M2_FE_plus,soni_Fusion,albina_MMD_v4
0,K1_TFIDF_LR,0.5319,0.1849,0.4853,0.7405
1,K2_CatBoost_38tab,-0.2422,0.2772,0.8615,0.8920
2,K3_ImageEmb32_LR,0.0279,0.2250,0.7354,0.8652
3,K4_ExpCombined_614,0.1985,0.1346,0.6751,0.8369
4,K5_Doc2Vec_LR,-0.0590,0.2202,0.7534,0.8762


## 7. Multi-seed robustness каждого ансамбля (10 random_state)

Для каждого ансамбля считаем mean ± std PR-AUC и R@P на 10 повторах group-aware split. Это даёт более устойчивую сравнительную оценку, чем single-seed.

In [15]:
def robustness(p_karina, label, n_seeds=10):
    X4 = np.column_stack([p_karina, p_diana_M2_FE_plus, p_soni_Fusion, p_albina_MMD_v4])
    prs, raps, rocs = [], [], []
    for s in range(n_seeds):
        g = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=s)
        fi, ei = next(g.split(X4, y_test, groups=sellers))
        sc = StandardScaler().fit(X4[fi])
        best = None
        for C in [0.01, 0.1, 1.0]:
            lr = LogisticRegression(C=C, class_weight='balanced', max_iter=2000, random_state=42)
            lr.fit(sc.transform(X4[fi]), y_test[fi])
            p_e = lr.predict_proba(sc.transform(X4[ei]))[:, 1]
            pr = average_precision_score(y_test[ei], p_e)
            if best is None or pr > best['pr']:
                best = {'pr': pr, 'rap': rap90(y_test[ei], p_e), 'roc': roc_auc_score(y_test[ei], p_e)}
        prs.append(best['pr']); raps.append(best['rap']); rocs.append(best['roc'])
    prs, raps, rocs = np.array(prs), np.array(raps), np.array(rocs)
    return {'label': label, 'pr_mean': prs.mean(), 'pr_std': prs.std(),
             'rap_mean': raps.mean(), 'rap_std': raps.std(),
             'roc_mean': rocs.mean(), 'pr_min': prs.min(), 'pr_max': prs.max()}

robust = [robustness(p, lbl) for (lbl, p) in variants]
rows = [{'mobile/ads канал': r['label'],
         'PR mean': r['pr_mean'], 'PR std': r['pr_std'], 'PR min': r['pr_min'], 'PR max': r['pr_max'],
         'R@P mean': r['rap_mean'], 'R@P std': r['rap_std'], 'ROC mean': r['roc_mean']}
         for r in robust]
df_robust = pd.DataFrame(rows)
df_robust.round(4)

,mobile/ads канал,PR mean,PR std,PR min,PR max,R@P mean,R@P std,ROC mean
0,K1_TFIDF_LR,0.7387,0.0241,0.7017,0.7761,0.0954,0.0267,0.9577
1,K2_CatBoost_38tab,0.7430,0.0305,0.6899,0.7977,0.1664,0.0555,0.9555
2,K3_ImageEmb32_LR,0.7423,0.0297,0.7025,0.7952,0.1406,0.0709,0.9565
3,K4_ExpCombined_614,0.7379,0.0287,0.7005,0.7888,0.1147,0.0531,0.9569
4,K5_Doc2Vec_LR,0.7367,0.0298,0.6964,0.7965,0.1037,0.0672,0.9520


## 8. Сохранение probas и итоговая сводка

In [16]:
OUT = NB_DIR / 'karina_extended_channels'
OUT.mkdir(exist_ok=True)
np.save(OUT / 'p_K1_TFIDF_LR.npy',         p_K1_tfidf_lr.astype('float32'))
np.save(OUT / 'p_K2_CatBoost_38tab.npy',   p_K2_cb_tab.astype('float32'))
np.save(OUT / 'p_K3_ImageEmb32_LR.npy',    p_K3_img_lr.astype('float32'))
np.save(OUT / 'p_K4_ExpCombined_614.npy',  p_K4_exp_combined.astype('float32'))
np.save(OUT / 'p_K5_Doc2Vec_LR.npy',       p_K5_doc2vec_lr.astype('float32'))
for e in ensembles:
    np.save(OUT / f'ensemble_with_{e["label"]}.npy', e['p_full'].astype('float32'))
print(f'все probas сохранены в {OUT}')

все probas сохранены в real_estate_approaches/notebooks/karina_extended_channels


In [17]:
leader_by_pr_robust = max(robust, key=lambda r: r['pr_mean'])
leader_by_rap_robust = max(robust, key=lambda r: r['rap_mean'])
print(f'Лидер по mean PR-AUC (10 seeds): {leader_by_pr_robust["label"]} → {leader_by_pr_robust["pr_mean"]:.4f} ± {leader_by_pr_robust["pr_std"]:.4f}')
print(f'Лидер по mean R@P (10 seeds):    {leader_by_rap_robust["label"]} → {leader_by_rap_robust["rap_mean"]:.4f} ± {leader_by_rap_robust["rap_std"]:.4f}')

Лидер по mean PR-AUC (10 seeds): K2_CatBoost_38tab → 0.7430 ± 0.0305
Лидер по mean R@P (10 seeds):    K2_CatBoost_38tab → 0.1664 ± 0.0555
